In [2]:
!nvidia-smi

Fri Mar  6 21:01:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
!pwd

/content


In [23]:
%cd /content
!rm -rf ./Continual/

/content


In [24]:
!git clone https://github.com/nnminh322/Continual.git
!pip install uv
%cd Continual/gainlora_baseline_origin
!bash setup_kaggle_colab.sh

Cloning into 'Continual'...
remote: Enumerating objects: 1532, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 1532 (delta 36), reused 38 (delta 19), pack-reused 1475 (from 1)
Receiving objects: 100% (1532/1532), 29.26 MiB | 10.54 MiB/s, done.
Resolving deltas: 100% (652/652), done.
Updating files: 100% (1597/1597), done.
/content/Continual/gainlora_baseline_origin
SpecRoute Setup for Kaggle/Colab
[Env] Detected: colab
[Setup] Using current Python environment

[Remove] Uninstalling conflicting packages...
Found existing installation: torch 2.5.1+cu121
Uninstalling torch-2.5.1+cu121:
  Successfully uninstalled torch-2.5.1+cu121
Found existing installation: torchvision 0.20.1+cu121
Uninstalling torchvision-0.20.1+cu121:
  Successfully uninstalled torchvision-0.20.1+cu121
Found existing installation: torchaudio 2.5.1+cu121
Uninstalling torchaudio-2.5.1+cu121:
  Successfully uninstalled torchaudio-2.5.1+cu121
Found existing

In [20]:
import subprocess, sys
# Use subprocess to avoid kernel's stale numpy/jax modules
result = subprocess.run([sys.executable, "-c", """
import torch, transformers, datasets, numpy, scipy, sys
print(f"python:       {sys.version.split()[0]}")
print(f"torch:        {torch.__version__}")
print(f"cuda:         {torch.cuda.is_available()}, devices={torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"gpu0:         {torch.cuda.get_device_name(0)}")
print(f"transformers: {transformers.__version__}")
print(f"datasets:     {datasets.__version__}")
print(f"numpy:        {numpy.__version__}")
print(f"scipy:        {scipy.__version__}")
try:
    import cupy; print(f"cupy:         {cupy.__version__}")
except: print("cupy:         NOT INSTALLED")
# Verify torch version is correct
assert torch.__version__.startswith("2.5.1"), f"WRONG TORCH: {torch.__version__}"
assert numpy.__version__ == "1.26.4", f"WRONG NUMPY: {numpy.__version__}"
print("\\nAll version checks PASSED")
"""], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("Version check failed!")
import os; print(f"cwd: {os.getcwd()}")

python:       3.12.12
torch:        2.5.1+cu121
cuda:         True, devices=1
gpu0:         Tesla T4
transformers: 4.40.2
datasets:     2.21.0
numpy:        1.26.4
scipy:        1.14.1
cupy:         13.6.0

All version checks PASSED

cwd: /content/Continual/gainlora_baseline_origin


In [9]:
!bash gen_script_superni_order1_t5_specroute.sh 0 google-t5/t5-large

[GPU] Detected T4 GPUs (15360MB VRAM each)
[GPU] Strategy: 1x T4 + fp16
[GPU] Using CUDA_VISIBLE_DEVICES=0

2026-03-06 21:16:40.596569: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772831800.835922    7090 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772831800.899183    7090 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772831801.391203    7090 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772831801.391240    7090 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same ta

In [10]:
import subprocess
# Check for errors / tracebacks in the most recent training output
# Also check what files were produced
print("=== Checking for errors in output logs ===")
r = subprocess.run("find logs_and_outputs -name '*.out' -o -name '*.log' 2>/dev/null | head -5", shell=True, capture_output=True, text=True, cwd="/content/Continual/gainlora_baseline_origin")
print("Log files found:", r.stdout or "NONE")
print()
# Check if output directories were created
r2 = subprocess.run("ls -la logs_and_outputs/gen_script_superni_order1_t5_specroute/outputs/ 2>&1 | head -20", shell=True, capture_output=True, text=True, cwd="/content/Continual/gainlora_baseline_origin")
print("=== Output directories ===")
print(r2.stdout or r2.stderr)

=== Checking for errors in output logs ===
Log files found: NONE

=== Output directories ===
ls: cannot access 'logs_and_outputs/gen_script_superni_order1_t5_specroute/outputs/': No such file or directory



In [ ]:
%%bash
cd /content/Continual/gainlora_baseline_origin
python src/run_t5.py \
   --do_train --do_predict --predict_with_generate \
   --model_name_or_path google-t5/t5-large \
   --data_dir CL_Benchmark \
   --task_order task1572_samsum_summary,task363_sst2_polarity_classification \
   --task_config_dir configs/gen_script_superni_order1_t5_configs/task1572_samsum_summary \
   --output_dir /tmp/test_specroute \
   --per_device_train_batch_size 2 --per_device_eval_batch_size 2 \
   --gradient_accumulation_steps 1 --learning_rate 0.0003 --num_train_epochs 1 \
   --max_source_length 512 --max_target_length 50 --generation_max_length 50 \
   --add_task_name False --add_dataset_name False \
   --overwrite_output_dir --overwrite_cache \
   --lr_scheduler_type constant --warmup_steps 0 \
   --logging_strategy steps --logging_steps 10 --save_total_limit 1 \
   --lora_r 4 --lora_alpha 32 --lora_dropout 0.0 \
   --run_single True --data_replay_freq -1 --mlp_hidden_dim 100 \
   --model_name specroute --threshold 0.995 --transthreshold 0.995 --fp16 \
   2>&1 | grep -iE "error|trace|exception|File " | head -15

decoder.block.21.layer.1.EncDecAttention.lora_q.lora_B
decoder.block.21.layer.1.EncDecAttention.lora_v.lora_B
decoder.block.22.layer.0.SelfAttention.lora_q.lora_B
decoder.block.22.layer.0.SelfAttention.lora_v.lora_B
decoder.block.22.layer.1.EncDecAttention.lora_q.lora_B
decoder.block.22.layer.1.EncDecAttention.lora_v.lora_B
decoder.block.23.layer.0.SelfAttention.lora_q.lora_B
decoder.block.23.layer.0.SelfAttention.lora_v.lora_B
decoder.block.23.layer.1.EncDecAttention.lora_q.lora_B
decoder.block.23.layer.1.EncDecAttention.lora_v.lora_B
Total number of parameters: 0.589M, rate: 0.08%
-----Gradient checkpointing: False -----
tmp
Traceback (most recent call last):
  File "/content/Continual/gainlora_baseline_origin/src/run_t5.py", line 977, in <module>
    main()
  File "/content/Continual/gainlora_baseline_origin/src/run_t5.py", line 844, in main
    trainer.get_reg_matrix()
  File "/content/Continual/gainlora_baseline_origin/src/cl_trainer_specroute.py", line 101, in get_reg_matrix
    